# 01 Encoders and the baseline you must beat

A frozen encoder plus a logistic regression is the cheapest decision model there is: one forward pass per message, no training of the encoder,
a probability per route. Everything later in the workshop has to beat this number, with an interval, or it is not an improvement.

We measure three encoders on the Meridian test split (200 messages, 40 scenarios of 5 variants each) and report accuracy with a Wilson interval
(which assumes 200 independent items) and a cluster-bootstrap interval (which respects that the 5 variants of one scenario are not independent).

In [1]:
import json, time, os
import numpy as np
import meridian as m
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

RESULTS = Path("data/results"); RESULTS.mkdir(parents=True, exist_ok=True)
print("hardware:", m.hardware())
tr, ca, te = m.load("train"), m.load("calibration"), m.load("test")
_, ytr = m.texts_labels(tr); _, yca = m.texts_labels(ca); _, yte = m.texts_labels(te)
cl_te = m.cluster_ids(te)

hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0


## Embed once, cache, and time it
`cached_embeddings` runs the frozen encoder over all 1,200 messages the first time and stores the vectors under `data/cache/`.
The time below is the cold run for the whole set at batch size 32.

In [2]:
emb, embed_seconds = {}, {}
for name in m.ENCODERS:
    t = time.perf_counter()
    try:
        emb[name] = m.cached_embeddings(name)
    except Exception as e:
        print(f"{name}: FAILED {type(e).__name__}: {str(e)[:200]}")
        continue
    embed_seconds[name] = time.perf_counter() - t
    print(f"{name:11s} dim {emb[name]['train'].shape[1]:4d}  all 1,200 messages in {embed_seconds[name]:6.1f} s (includes model load if not cached)")

bge-small   dim  384  all 1,200 messages in    0.0 s (includes model load if not cached)
modernbert  dim  768  all 1,200 messages in    0.1 s (includes model load if not cached)
lfm2.5-enc  dim 1024  all 1,200 messages in    0.0 s (includes model load if not cached)


## Logistic regression on frozen embeddings
The encoder stays frozen: it only turns each message into a vector (the `dim` printed above). A logistic regression then learns, from the 800 training vectors, a weight vector per route and returns a probability for each of the eight routes. Nothing inside the encoder is trained.

The next cell does five things for each of the three encoders:

1. **Choose `C` on the calibration split.** `C` is the inverse of the regularization strength: a small `C` pulls the weights toward zero (a simpler model that can underfit), a large `C` lets them fit the training vectors closely. `fit_lr` tries five values and keeps the one with the best calibration accuracy (on a tie, the smaller `C`). The test split is never used for this choice, so the test accuracy is not tuned.
2. **Predict the 200 test messages.** `correct` marks each message right or wrong, `k` counts the right ones, `n` is 200.
3. **Wilson interval.** A 95% interval for the accuracy `k / n`, computed from `k` and `n` alone. It treats the 200 messages as independent.
4. **Cluster bootstrap interval.** The test set is 40 scenarios of 5 variants, and variants of one scenario tend to be right or wrong together. The bootstrap resamples whole scenarios (with replacement, 5,000 times), recomputes the accuracy each time, and reports the 2.5th and 97.5th percentiles. It is the interval that respects the dependence.
5. **Macro-F1.** The F1 score of each route, averaged without weighting by route size, so a weak route counts as much as a strong one.

In [3]:
def fit_lr(X_tr, y_tr, X_ca, y_ca):
    best = None
    for C in (0.1, 1, 10, 100, 1000):
        clf = LogisticRegression(C=C, max_iter=2000).fit(X_tr, y_tr)
        acc = (clf.predict(X_ca) == y_ca).mean()
        if best is None or acc > best[0]:
            best = (acc, C, clf)
    return best

models, report = {}, {}
for name, E in emb.items():
    acc_ca, C, clf = fit_lr(E["train"], ytr, E["calibration"], yca)
    pred = clf.predict(E["test"])
    correct = (pred == yte)
    k, n = int(correct.sum()), len(yte)
    lo, hi = m.wilson(k, n)
    clo, chi = m.cluster_bootstrap_ci(correct.astype(float), cl_te)
    models[name] = clf
    report[name] = dict(C=C, k=k, n=n, acc=k/n, wilson=[lo, hi], cluster_ci=[clo, chi],
                        macro_f1=float(f1_score(yte, pred, average="macro")))
    print(f"{name:11s} C={C:<6} acc {k/n:.3f}  Wilson [{lo:.3f}, {hi:.3f}]  cluster bootstrap [{clo:.3f}, {chi:.3f}]  macro-F1 {report[name]['macro_f1']:.3f}")

bge-small   C=1      acc 0.970  Wilson [0.936, 0.986]  cluster bootstrap [0.930, 0.995]  macro-F1 0.970


modernbert  C=100    acc 0.925  Wilson [0.880, 0.954]  cluster bootstrap [0.885, 0.960]  macro-F1 0.925


lfm2.5-enc  C=10     acc 0.925  Wilson [0.880, 0.954]  cluster bootstrap [0.865, 0.975]  macro-F1 0.925


### Reading the output
Each line is one encoder: the `C` that won on the calibration split, the test accuracy, the two intervals, and the macro-F1. In our run:

- `bge-small` reaches 0.970 (Wilson 0.936 to 0.986). `modernbert` and `lfm2.5-enc` both reach 0.925 (Wilson 0.880 to 0.954).
- The intervals overlap, so this table alone does not show that one encoder is better; that needs a paired test on the same messages. Read it as the baseline to beat, not as a ranking.
- Where the two intervals differ, the cluster bootstrap is the one to quote, because it counts 40 independent scenarios and not 200. For `lfm2.5-enc` it runs from 0.865 to 0.975, wider than the Wilson interval.
- Macro-F1 sits close to accuracy because the test split has 25 messages per route. With unbalanced routes the two would drift apart, and macro-F1 would show the weak routes.

## Where the errors go
Per-route recall for the best encoder, and which route each wrong message went to.

In [4]:
best = max(report, key=lambda n: report[n]["acc"])
pred = models[best].predict(emb[best]["test"])
print("best frozen encoder:", best)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(yte, pred, labels=range(len(m.ROUTES)))
for i, r in enumerate(m.ROUTES):
    recall = cm[i, i] / cm[i].sum()
    wrong = {m.ROUTES[j]: int(cm[i, j]) for j in range(len(m.ROUTES)) if j != i and cm[i, j]}
    print(f"{r:22s} recall {recall:.2f}  wrong -> {wrong}")
report["_best"] = best

best frozen encoder: bge-small
baggage_claim          recall 1.00  wrong -> {}
cancellation_request   recall 1.00  wrong -> {}
out_of_scope           recall 0.96  wrong -> {'rebooking_request': 1}
payment_dispute        recall 1.00  wrong -> {}
rebooking_request      recall 0.92  wrong -> {'refund_claim': 1, 'service_complaint': 1}
refund_claim           recall 0.88  wrong -> {'payment_dispute': 3}
service_complaint      recall 1.00  wrong -> {}
visa_entry_question    recall 1.00  wrong -> {}


## Latency per message on this CPU
One message at a time (what a live router sees), 100 test messages after 3 warm-up calls. The hardware line is part of the result.

In [5]:
import torch
lat = {}
sample = [r["text"] for r in te[:100]]
for name in emb:
    loaded = m.load_encoder(name)
    stats = m.time_calls(lambda t: m.encode(name, [t], _loaded=loaded), sample)
    lat[name] = stats
    print(f"{name:11s} median {stats['median_ms']:6.1f} ms   p95 {stats['p95_ms']:6.1f} ms   (n={stats['n']}, batch 1)")
    del loaded
report["_latency_batch1"] = lat
report["_hardware"] = m.hardware()

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bge-small   median   31.5 ms   p95   99.0 ms   (n=100, batch 1)


modernbert  median  145.3 ms   p95  196.4 ms   (n=100, batch 1)


lfm2.5-enc  median  244.9 ms   p95  362.1 ms   (n=100, batch 1)


## Long inputs: what does 8,192 tokens cost on a CPU?
Meridian messages are 8 to 51 words, so the data cannot tell you. Here we repeat test messages until the input reaches a target length
and time one forward pass (3 repeats, median; inputs are skipped once one pass exceeds 60 s). The vendor claim (LFM2.5-Encoder 3.3 times faster than ModernBERT-base at 8k tokens on CPU) is Liquid AI's own
measurement on their hardware; this cell is ours, on the machine named above.

In [6]:
def long_text(tok, n_tokens):
    s = " ".join(r["text"] for r in tr)
    ids = tok(s, add_special_tokens=False)["input_ids"][:n_tokens]
    return tok.decode(ids)

long_res = {}
BUDGET_S = 300   # skip longer inputs once one forward pass takes more than this
REPEAT_S = 60    # a pass slower than this is measured once, not three times
MAX_LEN = int(os.environ.get("LONG_CONTEXT_MAX", 8192))   # set to 1024 on a machine with under 8 GB free: 8,192 tokens needs about 4 GB
load_start = os.getloadavg()
for name in ("modernbert", "lfm2.5-enc"):
    if name not in emb: continue
    tok, mod = m.load_encoder(name)
    long_res[name] = {}
    slow = False
    for L in [x for x in (256, 1024, 4096, 8192) if x <= MAX_LEN]:
        if slow:
            print(f"{name:11s} {L:5d} tokens  skipped (a shorter input already took more than {BUDGET_S} s)"); continue
        text = long_text(tok, L)
        b = tok([text], truncation=True, max_length=L, return_tensors="pt")
        actual = int(b["input_ids"].shape[1])
        ts = []
        with torch.inference_mode():
            for _ in range(3):
                t = time.perf_counter(); mod(**b); ts.append(time.perf_counter() - t)
                if ts[-1] > BUDGET_S: slow = True; break
                if ts[-1] > REPEAT_S: break
        long_res[name][L] = dict(tokens=actual, median_s=float(np.median(ts)), repeats=len(ts))
        print(f"{name:11s} {actual:5d} tokens  median {np.median(ts):7.2f} s  ({len(ts)} runs)")
    del mod
report["_long_context"] = long_res
report["_load_average_long_context"] = dict(start=list(load_start), end=list(os.getloadavg()))

Token indices sequence length is longer than the specified maximum sequence length for this model (27166 > 8192). Running this sequence through the model will result in indexing errors


modernbert    256 tokens  median    0.60 s  (3 runs)


modernbert   1024 tokens  median    3.46 s  (3 runs)


modernbert   4096 tokens  median   40.24 s  (3 runs)


modernbert   8192 tokens  median  196.40 s  (1 runs)


lfm2.5-enc    256 tokens  median    0.73 s  (3 runs)


lfm2.5-enc   1024 tokens  median    3.22 s  (3 runs)


lfm2.5-enc   4096 tokens  median   19.51 s  (3 runs)


lfm2.5-enc   8192 tokens  median   92.85 s  (1 runs)


In [7]:
(RESULTS / "01_encoders_and_baseline.json").write_text(json.dumps(report, indent=2, default=float))
print("saved", RESULTS / "01_encoders_and_baseline.json")

saved data/results/01_encoders_and_baseline.json
